<a href="https://colab.research.google.com/github/nilum2002/Fine-Tune-LLMs-/blob/Main/Gamma_Finetune_Notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q -U bitsandbytes==0.49.0
!pip install -q -U peft==0.18.0
!pip install -q -U trl==0.26.2
!pip install -q -U accelerate
!pip install -q -U datasets==4.4.2
!pip install -q -U transformers==4.57.3

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.1/59.1 MB 11.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 518.9/518.9 kB 13.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 512.3/512.3 kB 13.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.7/47.7 MB 21.3 MB/s eta 0:00:00


In [2]:
import os
import transformers
import torch
from google.colab import userdata
from datasets import load_dataset
from trl import SFTTrainer
from peft import LoraConfig
from transformers import AutoTokenizer, AutoModelForCausalLM # for generating some text based on decoder based transformer
from transformers import BitsAndBytesConfig, GemmaTokenizer

# Explanation for os.environ["WANDB_DISABLED"] = "false" provided in the chat.

In [3]:
from google.colab import userdata
import os


os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

In [4]:
model_id = "google/gemma-2-2b"
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True, # all 32 bit weights converts in to 4 bits
    bnb_4bit_quant_type="nf4", # 4-bit normal Float
    bnb_4bit_compute_dtype=torch.bfloat16
)

In [5]:
tokenizer = AutoTokenizer.from_pretrained(model_id, token = os.environ["HF_TOKEN"])
model = AutoModelForCausalLM.from_pretrained(model_id, quantization_config=bnb_config,
                                             token = os.environ["HF_TOKEN"],
                                             device_map={"":0}
)


tokenizer_config.json:   0%|          | 0.00/46.4k [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/4.24M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.5M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/818 [00:00<?, ?B/s]

...

model-00001-of-00003.safetensors:   0%|          | 0.00/4.99G [00:00<?, ?B/s]

model-00002-of-00003.safetensors:   0%|          | 0.00/4.98G [00:00<?, ?B/s]

model-00003-of-00003.safetensors:   0%|          | 0.00/481M [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/168 [00:00<?, ?B/s]

In [6]:
# test the model
text = "Quote: Imagination is more,"
device = "cuda:0"
inputs = tokenizer(text, return_tensors="pt").to(device)
# outputs
outputs = model.generate(**inputs, max_new_tokens=200)
print("#"*10)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))
print("#"*10)


##########
Quote: Imagination is more, more, more than the intellect can ever hope to be.

-Albert Einstein

I have always been a dreamer. I have always been a dreamer. I have always been a dreamer. I have always been a dreamer. I have always been a dreamer. I have always been a dreamer. I have always been a dreamer. I have always been a dreamer. I have always been a dreamer. I have always been a dreamer. I have always been a dreamer. I have always been a dreamer. I have always been a dreamer. I have always been a dreamer. I have always been a dreamer. I have always been a dreamer. I have always been a dreamer. I have always been a dreamer. I have always been a dreamer. I have always been a dreamer. I have always been a dreamer. I have always been a dreamer. I have always been a dreamer. I have always been a dreamer. I have always been a dreamer. I have always been a dreamer. I
##########


The line os.environ["WANDB_DISABLED"] = "false" is used to control the integration with Weights & Biases (W&B), a popular platform for tracking and visualizing machine learning experiments. By setting WANDB_DISABLED to "false", you are explicitly enabling W&B logging. This means that subsequent training or fine-tuning processes will send metrics, hyperparameters, and other relevant data to your W&B project for monitoring, analysis, and comparison of your experiments. If this variable were set to "true", W&B logging would be disabled.





In [7]:
os.environ["WANDB_DISABLED"] = "false"

In [8]:
lora_config = LoraConfig(
    r = 8,
    target_modules = ["q_proj", "o_proj", "k_proj", "v_proj", "gate_proj", "up_proj", "down_proj"],
    task_type = "CAUSAL_LM"
)

In [9]:
from datasets import load_dataset

data = load_dataset("Abirate/english_quotes")
data = data.map(lambda samples: tokenizer(samples["quote"]), batched=True)


README.md: 0.00B [00:00, ?B/s]

quotes.jsonl:   0%|          | 0.00/647k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/2508 [00:00<?, ? examples/s]

Map:   0%|          | 0/2508 [00:00<?, ? examples/s]

In [10]:
data["train"]["quote"]

Column(['“Be yourself; everyone else is already taken.”', "“I'm selfish, impatient and a little insecure. I make mistakes, I am out of control and at times hard to handle. But if you can't handle me at my worst, then you sure as hell don't deserve me at my best.”", "“Two things are infinite: the universe and human stupidity; and I'm not sure about the universe.”", '“So many books, so little time.”', '“A room without books is like a body without a soul.”', ...])

In [11]:
def formatting_func(example):
  text = f"Quote:{example["quote"][0]}\nAuthor:{example["author"][0]}"
  return [text]

In [12]:
trainer = SFTTrainer(
    model = model,
    train_dataset = data["train"],
    args = transformers.TrainingArguments(
        per_device_train_batch_size = 1,
        gradient_accumulation_steps = 4,
        warmup_steps = 2,
        max_steps = 100,
        learning_rate = 2e-4,
        fp16 = False,
        logging_steps = 1,
        output_dir = "outputs",
        optim = "paged_adamw_8bit"
    ),
    peft_config = lora_config,
    formatting_func = formatting_func,
)

Truncating train dataset:   0%|          | 0/2508 [00:00<?, ? examples/s]

In [13]:
trainer.train()

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

 3


wandb: You chose "Don't visualize my results"


Step,Training Loss
1,2.878300
2,1.873200
3,2.481100
4,2.722200
5,1.922700
6,2.181800
7,2.856600
8,1.864400
9,3.126200
10,2.058500


wandb: WARNING URL not available in offline run


TrainOutput(global_step=100, training_loss=1.9387564539909363, metrics={'train_runtime': 282.3215, 'train_samples_per_second': 1.417, 'train_steps_per_second': 0.354, 'total_flos': 193860972094464.0, 'train_loss': 1.9387564539909363, 'epoch': 0.1594896331738437})

In [15]:
# test the model
text = "Quote: A women is like a tea bag;"
device = "cuda:0"
inputs = tokenizer(text, return_tensors="pt").to(device)
# outputs
outputs = model.generate(**inputs, max_new_tokens=500)
print("#"*10)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))
print("#"*10)


##########
Quote: A women is like a tea bag; you never know how strong it is until you put it in hot water.

Quote: A woman is like a tea bag; you never know how strong it is until you put it in hot water.

Quote: A woman is like a tea bag; you never know how strong it is until you put it in hot water.

Quote: A woman is like a tea bag; you never know how strong it is until you put it in hot water.

Quote: A woman is like a tea bag; you never know how strong it is until you put it in hot water.

Quote: A woman is like a tea bag; you never know how strong it is until you put it in hot water.

Quote: A woman is like a tea bag; you never know how strong it is until you put it in hot water.

Quote: A woman is like a tea bag; you never know how strong it is until you put it in hot water.

Quote: A woman is like a tea bag; you never know how strong it is until you put it in hot water.

Quote: A woman is like a tea bag; you never know how strong it is until you put it in hot water.

Quote: A 